# Tar file extraction

In [ ]:
import tarfile
import os

source_path = "/kaggle/input/datasets/dschettler8845/brats-2021-task1/BraTS2021_Training_Data.tar"
extract_path = "/kaggle/working/brats2021"

os.makedirs(extract_path, exist_ok=True)

with tarfile.open(source_path) as tar:
    tar.extractall(path=extract_path)

print("Extraction complete!")


# Necessary libraries

In [ ]:
# -------------------------------
# Libraries
# -------------------------------
import numpy as np
import nibabel as nib
import cv2
import os
import tensorflow as tf
from tensorflow.keras import layers
from tqdm import tqdm
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time


# Data Processing

* 3D to 2D image using flair and segmentation
* remove empty slice and small tumor and brain slice using tumor >200 and brain >1500
* crop the image to focus on brain using 5 padding
* resize the image 128x128x1
* normalize using min max normalization

* normalize [-1,1]
  

In [ ]:
# -------------------------------
# Preprocessing
# -------------------------------
IMG_SIZE = 128
extract_path = "/kaggle/working/brats2021"

def normalize(img):
    img = img.astype(np.float32)
    mean = np.mean(img)
    std = np.std(img)
    if std == 0:
        return img
    img = (img - mean) / std
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    img = img * 2.0 - 1.0
    return img.astype("float16")

tumor_images = []

patient_folders = sorted([
    f for f in os.listdir(extract_path)
    if os.path.isdir(os.path.join(extract_path, f)) and "BraTS" in f
])

print("Total patients:", len(patient_folders))

for patient in tqdm(patient_folders):
    patient_path = os.path.join(extract_path, patient)
    files = os.listdir(patient_path)

    flair_path = [f for f in files if "flair" in f][0]
    seg_path   = [f for f in files if "seg" in f][0]

    flair = nib.load(os.path.join(patient_path, flair_path)).get_fdata()
    seg   = nib.load(os.path.join(patient_path, seg_path)).get_fdata()

    start = int(flair.shape[2] * 0.2)
    end   = int(flair.shape[2] * 0.8)

    for i in range(start, end):
        tumor_pixels = np.sum(seg[:, :, i] > 0)
        brain_pixels = np.sum(flair[:, :, i] > 0)

        if tumor_pixels > 200 and brain_pixels > 15000:
            img_slice  = flair[:, :, i]
            coords = np.argwhere(img_slice > np.percentile(img_slice, 5))
            if coords.shape[0] == 0:
                continue

            y_min, x_min = coords.min(axis=0)
            y_max, x_max = coords.max(axis=0)

            pad = 10
            y_min = max(0, y_min - pad)
            x_min = max(0, x_min - pad)
            y_max = min(img_slice.shape[0], y_max + pad)
            x_max = min(img_slice.shape[1], x_max + pad)

            cropped_img  = img_slice[y_min:y_max, x_min:x_max]

            img  = cv2.resize(cropped_img,  (IMG_SIZE, IMG_SIZE))
            img  = normalize(img)
            img  = np.expand_dims(img, axis=-1)

            tumor_images.append(img)

tumor_images = np.array(tumor_images, dtype="float32")
print("Filtered Dataset Shape:", tumor_images.shape)


# See random preprocess image

In [ ]:
# last
import matplotlib.pyplot as plt
import numpy as np

num_images = 20

indices = np.random.choice(len(tumor_images), num_images, replace=False)

plt.figure(figsize=(7, 6))

for i, idx in enumerate(indices):
    plt.subplot(4, 5, i + 1)
    
    img = tumor_images[idx].squeeze()
    
    # convert from [-1,1] → [0,1]
    img = (img + 1) / 2
    
    plt.imshow(img, cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

num_images = 20

indices = np.random.choice(len(tumor_images), num_images, replace=False)

plt.figure(figsize=(7,6))

for i, idx in enumerate(indices):
    plt.subplot(4, 5, i + 1)
    
    img = tumor_images[idx].squeeze()
    
    # convert from [-1,1] → [0,1]
    img = (img + 1) / 2
    
    plt.imshow(img, cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

# Hyperparameters for GAN training

In [ ]:
# -------------------------------
# GAN Hyperparameters
# -------------------------------
BATCH_SIZE = 32
NOISE_DIM = 100
EPOCHS = 14 #15     
IMG_SIZE = 128


# Shuffle the dataset to take random images in each batch

In [ ]:
dataset = tf.data.Dataset.from_tensor_slices(tumor_images)
dataset = dataset.shuffle(5000).batch(BATCH_SIZE, drop_remainder=True)

# Generator
* use Batchnormalization
* ReLU
* Conv2DTranspose
* tanh activation in last layer
# Dense layer 512->256->128->64

* kernel size 4
* maxpolling 2

In [ ]:
def build_generator():

    model = tf.keras.Sequential()

    model.add(tf.keras.layers.Dense(8*8*512, use_bias=False, input_shape=(NOISE_DIM,)))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.ReLU())

    model.add(tf.keras.layers.Reshape((8,8,512)))

    model.add(tf.keras.layers.Conv2DTranspose(256,4,2,'same',use_bias=False))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.ReLU())

    model.add(tf.keras.layers.Conv2DTranspose(128,4,2,'same',use_bias=False))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.ReLU())

    model.add(tf.keras.layers.Conv2DTranspose(64,4,2,'same',use_bias=False))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.ReLU())

    model.add(tf.keras.layers.Conv2DTranspose(1,4,2,'same',activation='tanh'))

    return model

generator = build_generator()

# Critic
* Conb2D
* LeakyReLU 0.2
* Dense layer 64->128-?256->512

In [ ]:
def build_critic():

    img = tf.keras.layers.Input(shape=(128,128,1))

    x = tf.keras.layers.Conv2D(64,4,2,'same')(img)
    x = tf.keras.layers.LeakyReLU(0.2)(x)

    x = tf.keras.layers.Conv2D(128,4,2,'same')(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)

    x = tf.keras.layers.Conv2D(256,4,2,'same')(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)

    x = tf.keras.layers.Conv2D(512,4,2,'same')(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)

    x = tf.keras.layers.Flatten()(x)
    output = tf.keras.layers.Dense(1)(x)

    return tf.keras.Model(img, output)

critic = build_critic()

# Optimizer=Adam
* learning rate G=1e-4
* learning rate Critic=5e-5
* beta=0,0.9

In [ ]:
generator_optimizer = tf.keras.optimizers.Adam(5e-5, beta_1=0.0, beta_2=0.9)
critic_optimizer = tf.keras.optimizers.Adam(1e-4, beta_1=0.0, beta_2=0.9)

# Use this learning rate

In [ ]:
generator_optimizer = tf.keras.optimizers.Adam(1e-4, beta_1=0.0, beta_2=0.9)
critic_optimizer = tf.keras.optimizers.Adam(1e-4, beta_1=0.0, beta_2=0.9)

# Gradient penalty
# LAMDA = 10

In [ ]:
LAMBDA = 10

def gradient_penalty(real, fake):

    batch_size = tf.shape(real)[0]
    alpha = tf.random.uniform([batch_size,1,1,1],0.0,1.0)

    interpolated = alpha*real + (1-alpha)*fake

    with tf.GradientTape() as tape:
        tape.watch(interpolated)
        pred = critic(interpolated)

    grads = tape.gradient(pred, interpolated)
    norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=[1,2,3]))
    gp = tf.reduce_mean((norm-1.0)**2)

    return gp

# Training step
* critic = 5
* critic loss (mean)

In [ ]:
@tf.function
def train_step(real_images):

    batch_size = tf.shape(real_images)[0]

    for _ in range(3):

        noise = tf.random.normal([batch_size, NOISE_DIM])

        with tf.GradientTape() as tape:

            fake_images = generator(noise, training=True)

            real_output = critic(real_images, training=True)
            fake_output = critic(fake_images, training=True)

            gp = gradient_penalty(real_images, fake_images)

            critic_loss = (
                tf.reduce_mean(fake_output)
                - tf.reduce_mean(real_output)
                + LAMBDA * gp
            )

        grads = tape.gradient(critic_loss, critic.trainable_variables)
        critic_optimizer.apply_gradients(zip(grads, critic.trainable_variables))

    noise = tf.random.normal([batch_size, NOISE_DIM])

    with tf.GradientTape() as tape:

        fake_images = generator(noise, training=True)
        fake_output = critic(fake_images, training=True)

        generator_loss = -tf.reduce_mean(fake_output)

    grads = tape.gradient(generator_loss, generator.trainable_variables)
    generator_optimizer.apply_gradients(zip(grads, generator.trainable_variables))

    return generator_loss, critic_loss

# See generate image in each epoch 

In [ ]:
import matplotlib.pyplot as plt
from tqdm import tqdm
import time

def generate_and_plot(epoch):

    noise = tf.random.normal([8, NOISE_DIM])
    predictions = generator(noise, training=False)
    predictions = (predictions + 1) / 2

    plt.figure(figsize=(8,6))
    for i in range(8):
        plt.subplot(2,4,i+1)
        plt.imshow(predictions[i,:,:,0], cmap='gray')
        plt.axis('off')
    plt.suptitle(f"Epoch {epoch}")
    plt.show()

# Training
# each epoch time = 609.75 Sec

In [ ]:
for epoch in range(EPOCHS):

    start = time.time()
    progbar = tqdm(dataset, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for image_batch in progbar:

        g_loss, c_loss = train_step(image_batch)

        progbar.set_postfix({
            "G": f"{g_loss.numpy():.3f}",
            "C": f"{c_loss.numpy():.3f}"
        })

    # ✅ Save model every 5 epochs
    # if (epoch + 1) > 5:
    generator.save(f"generator_epoch_{epoch+1}.keras")
    print(f"Saved model at epoch {epoch+1}")

    clear_output(wait=True)
    print(f"Epoch {epoch+1} time: {time.time()-start:.2f} sec")

    generate_and_plot(epoch+1)

# Train another time

In [ ]:
for epoch in range(EPOCHS):

    start = time.time()
    progbar = tqdm(dataset, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for image_batch in progbar:

        g_loss, c_loss = train_step(image_batch)

        progbar.set_postfix({
            "G": f"{g_loss.numpy():.3f}",
            "C": f"{c_loss.numpy():.3f}"
        })
    clear_output(wait=True)
    print(f"Epoch {epoch+1} time: {time.time()-start:.2f} sec")

    generate_and_plot(epoch+1)

# Generate Fake 3000 images in 32 batch size

In [ ]:
import tensorflow as tf
import numpy as np
import os

generator = tf.keras.models.load_model("/kaggle/working/generator_epoch_6.keras")

In [ ]:
# -------------------------------
# Safe Batch Generation
# -------------------------------
NUM_GEN = 2000
GEN_BATCH = 16   # VERY IMPORTANT (16 or 32 safe)

generated_images = []

for i in range(0, NUM_GEN, GEN_BATCH):
    noise = tf.random.normal([GEN_BATCH, NOISE_DIM])
    fake_batch = generator(noise, training=False)

    fake_batch = (fake_batch + 1) / 2  # scale to [0,1]
    generated_images.append(fake_batch.numpy())

generated_images = np.concatenate(generated_images, axis=0)

print("Generated shape:", generated_images.shape)

# Save generated images

In [ ]:
import numpy as np

# Assuming generated_images is your GAN output (128x128x1)
np.save("WGANgenerated_images6.npy", generated_images)
print("Generated images saved successfully!")

# Epoch 9(16)

In [ ]:
# new
indices = np.random.choice(len(generated_images), 20, replace=False)

plt.figure(figsize=(7,6))

for i, idx in enumerate(indices):
    plt.subplot(4,5,i+1)
    
    img = generated_images[idx].squeeze()
    img = (img + 1) / 2
    
    plt.imshow(img, cmap='gray')
    plt.axis('off')

plt.show()

# Epoch 8(15)

In [ ]:
# new
indices = np.random.choice(len(generated_images), 20, replace=False)

plt.figure(figsize=(7,6))

for i, idx in enumerate(indices):
    plt.subplot(4,5,i+1)
    
    img = generated_images[idx].squeeze()
    img = (img + 1) / 2
    
    plt.imshow(img, cmap='gray')
    plt.axis('off')

plt.show()

# Epoch 7(14)

In [ ]:
# new
indices = np.random.choice(len(generated_images), 20, replace=False)

plt.figure(figsize=(7,6))

for i, idx in enumerate(indices):
    plt.subplot(4,5,i+1)
    
    img = generated_images[idx].squeeze()
    img = (img + 1) / 2
    
    plt.imshow(img, cmap='gray')
    plt.axis('off')

plt.show()

# Epoch 5(12)

In [ ]:
# new
indices = np.random.choice(len(generated_images), 20, replace=False)

plt.figure(figsize=(7,6))

for i, idx in enumerate(indices):
    plt.subplot(4,5,i+1)
    
    img = generated_images[idx].squeeze()
    img = (img + 1) / 2
    
    plt.imshow(img, cmap='gray')
    plt.axis('off')

plt.show()

# Epoch 4(11)

In [ ]:
# new
indices = np.random.choice(len(generated_images), 20, replace=False)

plt.figure(figsize=(7,6))

for i, idx in enumerate(indices):
    plt.subplot(4,5,i+1)
    
    img = generated_images[idx].squeeze()
    img = (img + 1) / 2
    
    plt.imshow(img, cmap='gray')
    plt.axis('off')

plt.show()

# Epoch 6(13)

In [ ]:
# new
indices = np.random.choice(len(generated_images), 20, replace=False)

plt.figure(figsize=(7,6))

for i, idx in enumerate(indices):
    plt.subplot(4,5,i+1)
    
    img = generated_images[idx].squeeze()
    img = (img + 1) / 2
    
    plt.imshow(img, cmap='gray')
    plt.axis('off')

plt.show()

# take 2000 real and 2000 fake images for Evaluation (FID) and IS

In [ ]:
NUM = 2000

real_images = tumor_images[:NUM]
real_images = (real_images + 1) / 2

# fake_images = generated_images[:NUM]

# Visual comparision on **Real** and ** Fake** images

# Version1

In [ ]:
# New
# Shuffle both
real_idx = np.random.choice(len(real_images), 8, replace=False)
fake_idx = np.random.choice(len(fake_images), 8, replace=False)

real_sample = real_images[real_idx]
fake_sample = fake_images[fake_idx]

plt.figure(figsize=(7,6))  # 🔥 bigger

# 🔵 Real images (TOP ROWS)
for i in range(8):
    plt.subplot(4,4,i+1)
    plt.imshow(real_sample[i,:,:,0], cmap='gray')
    plt.axis('off')

# 🟢 Fake images (BOTTOM ROWS)
for i in range(8):
    plt.subplot(4,4,i+9)
    plt.imshow(fake_sample[i,:,:,0], cmap='gray')
    plt.axis('off')

plt.suptitle("Top: Shuffled Real MRI | Bottom: Shuffled Generated MRI (DCGAN)", fontsize=20)
plt.tight_layout()
plt.show()

In [ ]:
# New
# Shuffle both
real_idx = np.random.choice(len(real_images), 8, replace=False)
fake_idx = np.random.choice(len(fake_images), 8, replace=False)

real_sample = real_images[real_idx]
fake_sample = fake_images[fake_idx]

plt.figure(figsize=(7,6))  # 🔥 bigger

# 🔵 Real images (TOP ROWS)
for i in range(8):
    plt.subplot(4,4,i+1)
    plt.imshow(real_sample[i,:,:,0], cmap='gray')
    plt.axis('off')

# 🟢 Fake images (BOTTOM ROWS)
for i in range(8):
    plt.subplot(4,4,i+9)
    plt.imshow(fake_sample[i,:,:,0], cmap='gray')
    plt.axis('off')

plt.suptitle("Top: Shuffled Real MRI | Bottom: Shuffled Generated MRI (DCGAN)", fontsize=20)
plt.tight_layout()
plt.show()

In [ ]:
# bew==# Shuffle both
real_idx = np.random.choice(len(real_images), 16, replace=False)
fake_idx = np.random.choice(len(fake_images), 16, replace=False)

real_sample = real_images[real_idx]
fake_sample = fake_images[fake_idx]

plt.figure(figsize=(16,8))  # 🔥 bigger

# 🔵 Real images (TOP ROWS)
for i in range(16):
    plt.subplot(4,8,i+1)
    plt.imshow(real_sample[i,:,:,0], cmap='gray')
    plt.axis('off')

# 🟢 Fake images (BOTTOM ROWS)
for i in range(16):
    plt.subplot(4,8,i+17)
    plt.imshow(fake_sample[i,:,:,0], cmap='gray')
    plt.axis('off')

plt.suptitle("Top: Shuffled Real MRI | Bottom: Shuffled Generated MRI (DCGAN)", fontsize=20)
plt.tight_layout()
plt.show()

In [ ]:
# Shuffle both
real_idx = np.random.choice(len(real_images), 16, replace=False)
fake_idx = np.random.choice(len(fake_images), 16, replace=False)

real_sample = real_images[real_idx]
fake_sample = fake_images[fake_idx]

plt.figure(figsize=(16,8))  # 🔥 bigger

# 🔵 Real images (TOP ROWS)
for i in range(16):
    plt.subplot(4,8,i+1)
    plt.imshow(real_sample[i,:,:,0], cmap='gray')
    plt.axis('off')

# 🟢 Fake images (BOTTOM ROWS)
for i in range(16):
    plt.subplot(4,8,i+17)
    plt.imshow(fake_sample[i,:,:,0], cmap='gray')
    plt.axis('off')

plt.suptitle("Top: Shuffled Real MRI | Bottom: Shuffled Generated MRI (DCGAN)", fontsize=20)
plt.tight_layout()
plt.show()

# Version2

In [ ]:
# New
# Shuffle both
real_idx = np.random.choice(len(real_images), 8, replace=False)
fake_idx = np.random.choice(len(fake_images), 8, replace=False)

real_sample = real_images[real_idx]
fake_sample = fake_images[fake_idx]

plt.figure(figsize=(7,6))  # 🔥 bigger

# 🔵 Real images (TOP ROWS)
for i in range(8):
    plt.subplot(4,4,i+1)
    plt.imshow(real_sample[i,:,:,0], cmap='gray')
    plt.axis('off')

# 🟢 Fake images (BOTTOM ROWS)
for i in range(8):
    plt.subplot(4,4,i+9)
    plt.imshow(fake_sample[i,:,:,0], cmap='gray')
    plt.axis('off')

plt.suptitle("Top: Shuffled Real MRI | Bottom: Shuffled Generated MRI (DCGAN)", fontsize=20)
plt.tight_layout()
plt.show()

In [ ]:
# New
# Shuffle both
real_idx = np.random.choice(len(real_images), 8, replace=False)
fake_idx = np.random.choice(len(fake_images), 8, replace=False)

real_sample = real_images[real_idx]
fake_sample = fake_images[fake_idx]

plt.figure(figsize=(7,6))  # 🔥 bigger

# 🔵 Real images (TOP ROWS)
for i in range(8):
    plt.subplot(4,4,i+1)
    plt.imshow(real_sample[i,:,:,0], cmap='gray')
    plt.axis('off')

# 🟢 Fake images (BOTTOM ROWS)
for i in range(8):
    plt.subplot(4,4,i+9)
    plt.imshow(fake_sample[i,:,:,0], cmap='gray')
    plt.axis('off')

plt.suptitle("Top: Shuffled Real MRI | Bottom: Shuffled Generated MRI (DCGAN)", fontsize=20)
plt.tight_layout()
plt.show()

In [ ]:
data = np.load("/kaggle/input/datasets/shantaakterid170/generated-wgan-gp/WGANgenerated_images_128.npy")

print("Shape:", data.shape)
print("Min:", data.min(), "Max:", data.max())

# Epoch 11

In [ ]:
data11 = np.load("/kaggle/input/datasets/shantaakterid170/wgan-gp-generated-images/WGANgenerated_images4.npy")

print("Shape:", data11.shape)
print("Min:", data11.min(), "Max:", data11.max())

# Epoch 12

In [ ]:
data12 = np.load("/kaggle/input/datasets/shantaakterid170/wgan-gp-generated-images/WGANgenerated_images5.npy")

print("Shape:", data12.shape)
print("Min:", data12.min(), "Max:", data12.max())

# Epoch 13

In [ ]:
data15 = np.load("/kaggle/input/datasets/shantaakterid170/wgan-gp-generated-images/WGANgenerated_images9.npy")

print("Shape:", data15.shape)
print("Min:", data15.min(), "Max:", data15.max())

In [ ]:
# data = np.load("WGANgenerated_images_128.npy")

print("Shape:", real_images.shape)
print("Min:", real_images.min(), "Max:", real_images.max())

In [ ]:
fake_images=data15

In [ ]:
# data = np.load("WGANgenerated_images_128.npy")

print("Shape:", fake_images.shape)
print("Min:", fake_images.min(), "Max:", fake_images.max())

# Generate images

# Version 1

In [ ]:
# new
indices = np.random.choice(len(fake_images), 20, replace=False)

plt.figure(figsize=(7,6))

for i, idx in enumerate(indices):
    plt.subplot(4,5,i+1)
    
    img = fake_images[idx].squeeze()
    img = (img + 1) / 2
    
    plt.imshow(img, cmap='gray')
    plt.axis('off')

plt.show()

In [ ]:
indices = np.random.choice(len(data), 20, replace=False)

plt.figure(figsize=(7,6))

for i, idx in enumerate(indices):
    plt.subplot(4,5,i+1)
    
    img = data[idx].squeeze()
    img = (img + 1) / 2
    
    plt.imshow(img, cmap='gray')
    plt.axis('off')

plt.show()

# Version2 Epoch=12

In [ ]:
indices = np.random.choice(len(fake_images), 20, replace=False)

plt.figure(figsize=(7,6))

for i, idx in enumerate(indices):
    plt.subplot(4,5,i+1)
    
    img = fake_images[idx].squeeze()
    img = (img + 1) / 2
    
    plt.imshow(img, cmap='gray')
    plt.axis('off')

plt.show()

In [ ]:
indices = np.random.choice(len(fake_images), 20, replace=False)

plt.figure(figsize=(7,6))

for i, idx in enumerate(indices):
    plt.subplot(4,5,i+1)
    
    img = fake_images[idx].squeeze()
    img = (img + 1) / 2
    
    plt.imshow(img, cmap='gray')
    plt.axis('off')

plt.show()

In [ ]:
indices = np.random.choice(len(fake_images), 20, replace=False)

plt.figure(figsize=(7,6))

for i, idx in enumerate(indices):
    plt.subplot(4,5,i+1)
    
    img = fake_images[idx].squeeze()
    img = (img + 1) / 2
    
    plt.imshow(img, cmap='gray')
    plt.axis('off')

plt.show()

In [ ]:
indices = np.random.choice(len(fake_images), 20, replace=False)

plt.figure(figsize=(7,6))

for i, idx in enumerate(indices):
    plt.subplot(4,5,i+1)
    
    img = fake_images[idx].squeeze()
    img = (img + 1) / 2
    
    plt.imshow(img, cmap='gray')
    plt.axis('off')

plt.show()

# FID Evaluation
* Convert Grayscale to RGB
* Resize 299x299
* process Real and Fake images

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.inception_v3 import preprocess_input

def preprocess_images(images):
    """
    Preprocess images for InceptionV3 FID calculation.
    Assumes images are in [0,1].
    Converts grayscale to RGB and resizes to 299x299.
    """
    # scale [0,1] → [0,255]
    images = images * 255.0
    
    # grayscale → RGB
    if images.shape[-1] == 1:
        images = np.repeat(images, 3, axis=-1)
    
    # resize to 299x299
    images = tf.image.resize(images, (299, 299)).numpy()
    
    # preprocess for InceptionV3
    return preprocess_input(images)

# Example usage
real_proc = preprocess_images(real_images[:2000])
fake_proc = preprocess_images(fake_images[:2000])

# import InceptionV3 model for FID calculation

In [ ]:
from tensorflow.keras.applications import InceptionV3

model = InceptionV3(include_top=False, pooling='avg', input_shape=(299,299,3))

# Predict features for Real and Fake images in small batch(32)

In [ ]:
real_features = model.predict(real_proc, batch_size=32, verbose=1)
fake_features = model.predict(fake_proc, batch_size=32, verbose=1)

# Version 1

In [ ]:
# last
from scipy import linalg

def calculate_fid(f1, f2):
    mu1, sigma1 = f1.mean(axis=0), np.cov(f1, rowvar=False)
    mu2, sigma2 = f2.mean(axis=0), np.cov(f2, rowvar=False)

    diff = mu1 - mu2
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)

    if np.iscomplexobj(covmean):
        covmean = covmean.real

    return diff.dot(diff) + np.trace(sigma1 + sigma2 - 2 * covmean)

fid_score = calculate_fid(real_features, fake_features)
print("🎯 FID Score:", fid_score)

# FID score calculation

# Version 2

In [ ]:
from scipy import linalg

def calculate_fid(f1, f2):
    mu1, sigma1 = f1.mean(axis=0), np.cov(f1, rowvar=False)
    mu2, sigma2 = f2.mean(axis=0), np.cov(f2, rowvar=False)

    diff = mu1 - mu2
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)

    if np.iscomplexobj(covmean):
        
        covmean = covmean.real

    return diff.dot(diff) + np.trace(sigma1 + sigma2 - 2 * covmean)

fid_score = calculate_fid(real_features, fake_features)
print("🎯 FID Score:", fid_score)

# I calculate for different train generate images.It is another calculation.

# Epoch 11

In [ ]:
from scipy import linalg

def calculate_fid(f1, f2):
    mu1, sigma1 = f1.mean(axis=0), np.cov(f1, rowvar=False)
    mu2, sigma2 = f2.mean(axis=0), np.cov(f2, rowvar=False)

    diff = mu1 - mu2
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)

    if np.iscomplexobj(covmean):
        covmean = covmean.real

    return diff.dot(diff) + np.trace(sigma1 + sigma2 - 2 * covmean)

fid_score = calculate_fid(real_features, fake_features)
print("🎯 FID Score:", fid_score)

# Epoch 12

In [ ]:
# last
from scipy import linalg

def calculate_fid(f1, f2):
    mu1, sigma1 = f1.mean(axis=0), np.cov(f1, rowvar=False)
    mu2, sigma2 = f2.mean(axis=0), np.cov(f2, rowvar=False)

    diff = mu1 - mu2
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)

    if np.iscomplexobj(covmean):
        covmean = covmean.real

    return diff.dot(diff) + np.trace(sigma1 + sigma2 - 2 * covmean)

fid_score = calculate_fid(real_features, fake_features)
print("🎯 FID Score:", fid_score)

# Epoch 13

In [ ]:
# last
from scipy import linalg

def calculate_fid(f1, f2):
    mu1, sigma1 = f1.mean(axis=0), np.cov(f1, rowvar=False)
    mu2, sigma2 = f2.mean(axis=0), np.cov(f2, rowvar=False)

    diff = mu1 - mu2
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)

    if np.iscomplexobj(covmean):
        covmean = covmean.real

    return diff.dot(diff) + np.trace(sigma1 + sigma2 - 2 * covmean)

fid_score = calculate_fid(real_features, fake_features)
print("🎯 FID Score:", fid_score)

# Epoch 14

In [ ]:
# last
from scipy import linalg

def calculate_fid(f1, f2):
    mu1, sigma1 = f1.mean(axis=0), np.cov(f1, rowvar=False)
    mu2, sigma2 = f2.mean(axis=0), np.cov(f2, rowvar=False)

    diff = mu1 - mu2
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)

    if np.iscomplexobj(covmean):
        covmean = covmean.real

    return diff.dot(diff) + np.trace(sigma1 + sigma2 - 2 * covmean)

fid_score = calculate_fid(real_features, fake_features)
print("🎯 FID Score:", fid_score)

# Epoch 15

In [ ]:
# last
from scipy import linalg

def calculate_fid(f1, f2):
    mu1, sigma1 = f1.mean(axis=0), np.cov(f1, rowvar=False)
    mu2, sigma2 = f2.mean(axis=0), np.cov(f2, rowvar=False)

    diff = mu1 - mu2
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)

    if np.iscomplexobj(covmean):
        covmean = covmean.real

    return diff.dot(diff) + np.trace(sigma1 + sigma2 - 2 * covmean)

fid_score = calculate_fid(real_features, fake_features)
print("🎯 FID Score:", fid_score)

# Epoch 16

In [ ]:
# last
from scipy import linalg

def calculate_fid(f1, f2):
    mu1, sigma1 = f1.mean(axis=0), np.cov(f1, rowvar=False)
    mu2, sigma2 = f2.mean(axis=0), np.cov(f2, rowvar=False)

    diff = mu1 - mu2
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)

    if np.iscomplexobj(covmean):
        covmean = covmean.real

    return diff.dot(diff) + np.trace(sigma1 + sigma2 - 2 * covmean)

fid_score = calculate_fid(real_features, fake_features)
print("🎯 FID Score:", fid_score)

# Preprocess for IS in the same way as FID

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.inception_v3 import preprocess_input

def preprocess_images(images):
    # [-1,1] → [0,255]
    # images = (images + 1.0) * 127.5
        
    images = images * 255.0

    
    # grayscale → RGB
    images = np.repeat(images, 3, axis=-1)
    
    # resize
    images = tf.image.resize(images, (299, 299)).numpy()
    
    return preprocess_input(images)

fake_proc = preprocess_images(fake_images)

# Import model for IS

In [ ]:
from tensorflow.keras.applications import InceptionV3

model = InceptionV3(include_top=True, weights='imagenet')

# Predict only fake images features

In [ ]:
preds = model.predict(fake_proc, batch_size=16, verbose=1)

# Calculate IS. Only for Fake images

# Version 1

In [ ]:
# new
import numpy as np
from scipy.stats import entropy

def calculate_inception_score(preds, splits=10):
    scores = []
    N = preds.shape[0]
    
    for i in range(splits):
        part = preds[i * N // splits : (i + 1) * N // splits]
        
        py = np.mean(part, axis=0)
        kl = part * (np.log(part + 1e-10) - np.log(py + 1e-10))
        kl_sum = np.mean(np.sum(kl, axis=1))
        
        scores.append(np.exp(kl_sum))
    
    return np.mean(scores), np.std(scores)

is_mean, is_std = calculate_inception_score(preds)

print(f"🎯 Inception Score: {is_mean:.4f} ± {is_std:.4f}")

# Version 2

In [ ]:
import numpy as np
from scipy.stats import entropy

def calculate_inception_score(preds, splits=10):
    scores = []
    N = preds.shape[0]
    
    for i in range(splits):
        part = preds[i * N // splits : (i + 1) * N // splits]
        
        py = np.mean(part, axis=0)
        kl = part * (np.log(part + 1e-10) - np.log(py + 1e-10))
        kl_sum = np.mean(np.sum(kl, axis=1))
        
        scores.append(np.exp(kl_sum))
    
    return np.mean(scores), np.std(scores)

is_mean, is_std = calculate_inception_score(preds)

print(f"🎯 Inception Score: {is_mean:.4f} ± {is_std:.4f}")

# Epoch 11

In [ ]:
import numpy as np
from scipy.stats import entropy

def calculate_inception_score(preds, splits=10):
    scores = []
    N = preds.shape[0]
    
    for i in range(splits):
        part = preds[i * N // splits : (i + 1) * N // splits]
        
        py = np.mean(part, axis=0)
        kl = part * (np.log(part + 1e-10) - np.log(py + 1e-10))
        kl_sum = np.mean(np.sum(kl, axis=1))
        
        scores.append(np.exp(kl_sum))
    
    return np.mean(scores), np.std(scores)

is_mean, is_std = calculate_inception_score(preds)

print(f"🎯 Inception Score: {is_mean:.4f} ± {is_std:.4f}")

# Epoch 12

In [ ]:
# new
import numpy as np
from scipy.stats import entropy

def calculate_inception_score(preds, splits=10):
    scores = []
    N = preds.shape[0]
    
    for i in range(splits):
        part = preds[i * N // splits : (i + 1) * N // splits]
        
        py = np.mean(part, axis=0)
        kl = part * (np.log(part + 1e-10) - np.log(py + 1e-10))
        kl_sum = np.mean(np.sum(kl, axis=1))
        
        scores.append(np.exp(kl_sum))
    
    return np.mean(scores), np.std(scores)

is_mean, is_std = calculate_inception_score(preds)

print(f"🎯 Inception Score: {is_mean:.4f} ± {is_std:.4f}")

# Epoch 13

In [ ]:
# new
import numpy as np
from scipy.stats import entropy

def calculate_inception_score(preds, splits=10):
    scores = []
    N = preds.shape[0]
    
    for i in range(splits):
        part = preds[i * N // splits : (i + 1) * N // splits]
        
        py = np.mean(part, axis=0)
        kl = part * (np.log(part + 1e-10) - np.log(py + 1e-10))
        kl_sum = np.mean(np.sum(kl, axis=1))
        
        scores.append(np.exp(kl_sum))
    
    return np.mean(scores), np.std(scores)

is_mean, is_std = calculate_inception_score(preds)

print(f"🎯 Inception Score: {is_mean:.4f} ± {is_std:.4f}")

# Epoch 14

In [ ]:
# new
import numpy as np
from scipy.stats import entropy

def calculate_inception_score(preds, splits=10):
    scores = []
    N = preds.shape[0]
    
    for i in range(splits):
        part = preds[i * N // splits : (i + 1) * N // splits]
        
        py = np.mean(part, axis=0)
        kl = part * (np.log(part + 1e-10) - np.log(py + 1e-10))
        kl_sum = np.mean(np.sum(kl, axis=1))
        
        scores.append(np.exp(kl_sum))
    
    return np.mean(scores), np.std(scores)

is_mean, is_std = calculate_inception_score(preds)

print(f"🎯 Inception Score: {is_mean:.4f} ± {is_std:.4f}")

# Epoch 15

In [ ]:
# new
import numpy as np
from scipy.stats import entropy

def calculate_inception_score(preds, splits=10):
    scores = []
    N = preds.shape[0]
    
    for i in range(splits):
        part = preds[i * N // splits : (i + 1) * N // splits]
        
        py = np.mean(part, axis=0)
        kl = part * (np.log(part + 1e-10) - np.log(py + 1e-10))
        kl_sum = np.mean(np.sum(kl, axis=1))
        
        scores.append(np.exp(kl_sum))
    
    return np.mean(scores), np.std(scores)

is_mean, is_std = calculate_inception_score(preds)

print(f"🎯 Inception Score: {is_mean:.4f} ± {is_std:.4f}")

# Epoch 16

In [ ]:
# new
import numpy as np
from scipy.stats import entropy

def calculate_inception_score(preds, splits=10):
    scores = []
    N = preds.shape[0]
    
    for i in range(splits):
        part = preds[i * N // splits : (i + 1) * N // splits]
        
        py = np.mean(part, axis=0)
        kl = part * (np.log(part + 1e-10) - np.log(py + 1e-10))
        kl_sum = np.mean(np.sum(kl, axis=1))
        
        scores.append(np.exp(kl_sum))
    
    return np.mean(scores), np.std(scores)

is_mean, is_std = calculate_inception_score(preds)

print(f"🎯 Inception Score: {is_mean:.4f} ± {is_std:.4f}")

# Some more images

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Select 8 random images
idx_real = np.random.choice(len(real_images), 8, replace=False)
idx_fake = np.random.choice(len(fake_images), 8, replace=False)

real_sample = real_images[idx_real]
fake_sample = fake_images[idx_fake]

# Plot
plt.figure(figsize=(7,6))

# -----------------------
# Real Images (Top Row)
# -----------------------
for i in range(8):
    plt.subplot(4, 4, i + 1)
    plt.imshow(real_sample[i].squeeze(), cmap='gray')
    plt.title("Real")
    plt.axis('off')

# -----------------------
# Fake Images (Bottom Row)
# -----------------------
for i in range(8):
    plt.subplot(4, 4, 8 + i + 1)
    plt.imshow(fake_sample[i].squeeze(), cmap='gray')
    plt.title("Fake")
    plt.axis('off')

plt.tight_layout()
plt.show()

# Version2

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Select 8 random images
idx_real = np.random.choice(len(real_images), 8, replace=False)
idx_fake = np.random.choice(len(fake_images), 8, replace=False)

real_sample = real_images[idx_real]
fake_sample = fake_images[idx_fake]

# Plot
plt.figure(figsize=(12, 6))

# -----------------------
# Real Images (Top Row)
# -----------------------
for i in range(8):
    plt.subplot(2, 8, i + 1)
    plt.imshow(real_sample[i].squeeze(), cmap='gray')
    plt.title("Real")
    plt.axis('off')

# -----------------------
# Fake Images (Bottom Row)
# -----------------------
for i in range(8):
    plt.subplot(2, 8, 8 + i + 1)
    plt.imshow(fake_sample[i].squeeze(), cmap='gray')
    plt.title("Fake")
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# new
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images), 20, replace=False)
samples = fake_images[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(7,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(denormalize(samples[i]).squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images), 20, replace=False)
samples = fake_images[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(7,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(denormalize(samples[i]).squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

# Epoch 11

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images), 20, replace=False)
samples = fake_images[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(7,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(denormalize(samples[i]).squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images), 20, replace=False)
samples = fake_images[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(7,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(denormalize(samples[i]).squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

# Epoch 12

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images), 20, replace=False)
samples = fake_images[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(7,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(denormalize(samples[i]).squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images), 20, replace=False)
samples = fake_images[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(7,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(denormalize(samples[i]).squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

# Epoch 13

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images), 20, replace=False)
samples = fake_images[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(7,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(denormalize(samples[i]).squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images), 20, replace=False)
samples = fake_images[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(7,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(denormalize(samples[i]).squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

# Epoch 14

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images), 20, replace=False)
samples = fake_images[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(7,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(denormalize(samples[i]).squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images), 20, replace=False)
samples = fake_images[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(7,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(denormalize(samples[i]).squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images), 20, replace=False)
samples = fake_images[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(7,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(denormalize(samples[i]).squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

# Epoch 15

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images), 20, replace=False)
samples = fake_images[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(7,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(denormalize(samples[i]).squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images), 20, replace=False)
samples = fake_images[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(7,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(denormalize(samples[i]).squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

# Epoch 16

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images), 20, replace=False)
samples = fake_images[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(7,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(denormalize(samples[i]).squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images), 20, replace=False)
samples = fake_images[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(7,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(denormalize(samples[i]).squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

# Some additional Code that I try . Not import. You can ignore.

In [ ]:
import numpy as np
import tensorflow as tf
from tqdm import tqdm
import cv2

# Number of images to generate
num_images = 2000
batch_size = 16  # small batch size
generated_images = []

for i in tqdm(range(0, num_images, batch_size)):
    current_batch_size = min(batch_size, num_images - i)
    noise = tf.random.normal([current_batch_size, NOISE_DIM])
    fake_images = generator(noise, training=False)  # generate images

    # Convert to numpy and rescale from [-1,1] to [0,255] if you want
    fake_images = (fake_images.numpy() + 1.0) / 2.0 * 255.0
    fake_images = fake_images.astype(np.uint8)

    # Optionally resize or squeeze channel
    fake_images = np.squeeze(fake_images, axis=-1)  # shape (batch, 128,128)
    generated_images.extend(fake_images)

generated_images = np.array(generated_images)
print("Generated images shape:", generated_images.shape)

In [ ]:
np.save("generated_tumor_images.npy", generated_images)

In [ ]:
import tensorflow as tf
import numpy as np
import cv2

# Function to prepare grayscale images for InceptionV3
def preprocess_for_inception(img_array):
    processed = []
    for img in img_array:
        # Resize to 299x299
        img_resized = cv2.resize(img, (299, 299))
        # Convert grayscale to RGB
        img_rgb = np.stack([img_resized]*3, axis=-1)
        # Scale to [-1,1] for InceptionV3
        img_rgb = img_rgb / 127.5 - 1.0
        processed.append(img_rgb)
    return np.array(processed, dtype=np.float32)

# Assume you already have:
# real_images -> shape (2000, 128, 128)
# generated_images -> shape (2000, 128, 128)

real_images_proc = preprocess_for_inception(real_images[:2000])
gen_images_proc  = preprocess_for_inception(generated_images[:2000])

In [ ]:
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input

# Load pre-trained InceptionV3
inception_model = InceptionV3(include_top=False, pooling='avg', input_shape=(299,299,3))

# Get features
real_features = inception_model.predict(real_images_proc, batch_size=32, verbose=1)
gen_features  = inception_model.predict(gen_images_proc, batch_size=32, verbose=1)

In [ ]:
from scipy.linalg import sqrtm
import numpy as np

def calculate_fid_stable(real_features, gen_features, eps=1e-6):
    mu1, sigma1 = np.mean(real_features, axis=0), np.cov(real_features, rowvar=False)
    mu2, sigma2 = np.mean(gen_features, axis=0), np.cov(gen_features, rowvar=False)

    # Add small epsilon to diagonal to prevent singularity
    sigma1 += np.eye(sigma1.shape[0]) * eps
    sigma2 += np.eye(sigma2.shape[0]) * eps

    covmean = sqrtm(sigma1.dot(sigma2))
    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fid = np.sum((mu1 - mu2)**2) + np.trace(sigma1 + sigma2 - 2*covmean)
    return fid

fid_score = calculate_fid_stable(real_features, gen_features)
print("FID score (stable):", fid_score)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from scipy import linalg

# -------------------------------
# 1️⃣ Preprocess Images
# -------------------------------
def preprocess_images(images):
    """
    Preprocess images for InceptionV3 FID calculation.
    Assumes images are in [0,1].
    Converts grayscale to RGB and resizes to 299x299.
    """
    # Scale [0,1] → [0,255]
    images = images * 255.0

    # Grayscale → RGB
    if images.shape[-1] == 1:
        images = np.repeat(images, 3, axis=-1)

    # Resize to 299x299
    images = tf.image.resize(images, (299, 299)).numpy()

    # Preprocess for InceptionV3
    return preprocess_input(images)

# Preprocess real and generated images
real_proc = preprocess_images(real_images[:2000])   # 2000 real images
fake_proc = preprocess_images(generated_images[:2000])  # 2000 fake images

# -------------------------------
# 2️⃣ Extract Inception Features
# -------------------------------
model = InceptionV3(include_top=False, pooling='avg', input_shape=(299,299,3))
real_features = model.predict(real_proc, batch_size=32, verbose=1)
fake_features = model.predict(fake_proc, batch_size=32, verbose=1)

# -------------------------------
# 3️⃣ Calculate FID
# -------------------------------
def calculate_fid(f1, f2, eps=1e-6):
    mu1, sigma1 = np.mean(f1, axis=0), np.cov(f1, rowvar=False)
    mu2, sigma2 = np.mean(f2, axis=0), np.cov(f2, rowvar=False)

    # Add small epsilon to diagonals to avoid singular matrix
    sigma1 += np.eye(sigma1.shape[0]) * eps
    sigma2 += np.eye(sigma2.shape[0]) * eps

    # Compute sqrt of product of covariances
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)
    if np.iscomplexobj(covmean):
        covmean = covmean.real

    diff = mu1 - mu2
    fid = diff.dot(diff) + np.trace(sigma1 + sigma2 - 2 * covmean)
    return fid

fid_score = calculate_fid(real_features, fake_features)
print("🎯 FID Score:", fid_score)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input
from scipy import linalg

In [ ]:
# data = np.load("WGANgenerated_images_128.npy")

print("Shape:", generated_images.shape)
print("Min:", generated_images.min(), "Max:", generated_images.max())

In [ ]:
# data = np.load("WGANgenerated_images_128.npy")

print("Shape:", real_images.shape)
print("Min:", real_images.min(), "Max:", real_images.max())

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input
from scipy import linalg


In [ ]:
def preprocess_images(images, img_range=(0,1)):
    """
    Preprocess images for InceptionV3 FID calculation.
    Converts grayscale to RGB and resizes to 299x299.
    img_range: tuple indicating input image scale
    """
    # Normalize to [0,255]
    min_val, max_val = img_range
    images = (images - min_val) / (max_val - min_val) * 255.0

    # Ensure channel dimension
    if len(images.shape) == 3:
        images = np.expand_dims(images, axis=-1)

    # Grayscale → RGB
    if images.shape[-1] == 1:
        images = np.repeat(images, 3, axis=-1)

    # Resize to 299x299
    images = tf.image.resize(images, (299, 299)).numpy()

    # Preprocess for InceptionV3
    return preprocess_input(images)

In [ ]:
# Preprocess real and generated images
real_proc = preprocess_images(real_images, img_range=(0,1))        # real images scaled [0,1]
fake_proc = preprocess_images(generated_images, img_range=(0,254)) # generated images scaled [0,254]


In [ ]:
# -------------------------------
# 2️⃣ Load InceptionV3 model
# -------------------------------
model = InceptionV3(include_top=False, pooling='avg', input_shape=(299,299,3))

# -------------------------------
# 3️⃣ Extract Features
# -------------------------------
real_features = model.predict(real_proc, batch_size=32, verbose=1)
fake_features = model.predict(fake_proc, batch_size=32, verbose=1)

In [ ]:
# -------------------------------
# 4️⃣ Calculate FID
# -------------------------------
def calculate_fid(f1, f2, eps=1e-6):
    mu1, sigma1 = np.mean(f1, axis=0), np.cov(f1, rowvar=False)
    mu2, sigma2 = np.mean(f2, axis=0), np.cov(f2, rowvar=False)

    # Add epsilon to diagonal to avoid singular matrices
    sigma1 += np.eye(sigma1.shape[0]) * eps
    sigma2 += np.eye(sigma2.shape[0]) * eps

    # Compute sqrt of product of covariances
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)
    if np.iscomplexobj(covmean):
        covmean = covmean.real

    diff = mu1 - mu2
    fid = diff.dot(diff) + np.trace(sigma1 + sigma2 - 2 * covmean)
    return fid

# Compute FID
fid_score = calculate_fid(real_features, fake_features)
print("🎯 FID Score:", fid_score)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input
from scipy.stats import entropy

# -------------------------------
# 1️⃣ Preprocess generated images
# -------------------------------
def preprocess_for_inception(images, img_range=(0,254)):
    """
    Preprocess images for Inception Score.
    Converts grayscale to RGB and resizes to 299x299.
    """
    # Normalize to [0,255]
    min_val, max_val = img_range
    images = (images - min_val) / (max_val - min_val) * 255.0

    # Ensure channel dimension
    if len(images.shape) == 3:
        images = np.expand_dims(images, axis=-1)

    # Grayscale → RGB
    if images.shape[-1] == 1:
        images = np.repeat(images, 3, axis=-1)

    # Resize to 299x299
    images = tf.image.resize(images, (299, 299)).numpy()

    # Preprocess for InceptionV3
    return preprocess_input(images)

gen_proc = preprocess_for_inception(generated_images, img_range=(0,254))

# -------------------------------
# 2️⃣ Load InceptionV3
# -------------------------------
model = InceptionV3(include_top=True, weights='imagenet', input_shape=(299,299,3))

# -------------------------------
# 3️⃣ Get class probabilities
# -------------------------------
# Output shape: (num_images, 1000)
preds = model.predict(gen_proc, batch_size=32, verbose=1)
preds = tf.nn.softmax(preds).numpy()  # make sure it's probabilities

# -------------------------------
# 4️⃣ Compute Inception Score
# -------------------------------
def inception_score(preds, splits=10):
    N = preds.shape[0]
    split_size = N // splits
    scores = []

    for i in range(splits):
        part = preds[i*split_size : (i+1)*split_size]
        py = np.mean(part, axis=0)
        kl_divs = [entropy(p, py) for p in part]
        split_score = np.exp(np.mean(kl_divs))
        scores.append(split_score)

    return np.mean(scores), np.std(scores)

mean_is, std_is = inception_score(preds)
print(f"🎯 Inception Score: {mean_is:.3f} ± {std_is:.3f}")

In [ ]:
import matplotlib.pyplot as plt
import random

# -------------------------------
# 1️⃣ Prepare images for display
# -------------------------------
# Pick 8 random real and 8 random generated images
real_samples = random.sample(list(real_images), 8)
fake_samples = random.sample(list(generated_images), 8)

# Normalize generated images to [0,1] for display
fake_samples_norm = [img/254.0 if img.max() > 1 else img for img in fake_samples]

# -------------------------------
# 2️⃣ Plot images
# -------------------------------
plt.figure(figsize=(16, 4))

# Real images
for i, img in enumerate(real_samples):
    plt.subplot(2, 8, i+1)
    plt.imshow(img.squeeze(), cmap='gray')
    plt.title("Real")
    plt.axis('off')

# Generated images
for i, img in enumerate(fake_samples_norm):
    plt.subplot(2, 8, i+9)
    plt.imshow(img.squeeze(), cmap='gray')
    plt.title("Fake")
    plt.axis('off')

plt.tight_layout()
plt.show()